In [1]:
import torch
from transformers import T5ForConditionalGeneration, BertTokenizer
import torchvision.models as models
import torch.nn as nn

class R2GenModel(torch.nn.Module):
    def __init__(self, model_name='t5-small', device='cuda', dropout_prob=0.1):
        super(R2GenModel, self).__init__()
        self.tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
        self.model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)
        self.device = device

        # Add visual extractor (ResNet101)
        self.visual_extractor = models.resnet101(pretrained=True).to(device)
        self.visual_extractor.fc = torch.nn.Linear(self.visual_extractor.fc.in_features, self.model.config.d_model).to(device)

        # Dropout layer
        self.dropout = nn.Dropout(dropout_prob)

    def forward(self, input_ids, attention_mask, images, labels=None):
        visual_features = self.extract_visual_features(images)
        visual_features = self.dropout(visual_features)  # Apply dropout
        if labels is not None:
            return self.model(input_ids=input_ids, attention_mask=attention_mask, encoder_outputs=(visual_features,), labels=labels)
        else:
            return self.model.generate(input_ids=input_ids, attention_mask=attention_mask, encoder_outputs=(visual_features,))

    def extract_visual_features(self, images):
        images = images.to(self.device)
        # Use features from the second-to-last layer for pooling
        visual_features = self.visual_extractor(images)
        visual_features = visual_features.unsqueeze(1)  # Add sequence dimension
        batch_size, dim = visual_features.size(0), visual_features.size(2)
        visual_features = visual_features.expand(batch_size, 512, dim)  # Expand to the same sequence length
        return visual_features

    def generate_caption(self, input_ids, images, max_length=50):
        visual_features = self.extract_visual_features(images)
        generated_ids = self.model.generate(input_ids=input_ids, encoder_outputs=(visual_features,), max_length=max_length, num_beams=1)
        generated_texts = [self.tokenizer.decode(g, skip_special_tokens=True, clean_up_tokenization_spaces=True) for g in generated_ids]
        return generated_texts


c:\Users\vijay\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torch
from PIL import Image
from torchvision import transforms

class ReportImageDataset(torch.utils.data.Dataset):
    def __init__(self, reports, image_paths, tokenizer, image_transform=None):
        self.reports = reports
        self.image_paths = image_paths
        self.tokenizer = tokenizer
        self.image_transform = image_transform

    def __len__(self):
        return len(self.reports)

    def __getitem__(self, idx):
        report = self.reports[idx]
        image_path = self.image_paths[idx]

        # Load and preprocess image
        image = Image.open(image_path).convert('RGB')
        if self.image_transform:
            image = self.image_transform(image)

        # Tokenize report text
        inputs = self.tokenizer.encode_plus("generate report: " + report, return_tensors="pt", max_length=512, truncation=True, padding="max_length")

        return {
            'input_ids': inputs['input_ids'].flatten(),
            'attention_mask': inputs['attention_mask'].flatten(),
            'image': image
        }

In [3]:
def train(model, train_dataset, val_dataset, batch_size=8, num_epochs=5, learning_rate=1e-4):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_dataloader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

    model.to(device)
    model.train()

    for epoch in range(num_epochs):
        total_train_loss = 0.0
        model.train()
        for batch in train_dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            images = batch['image'].to(device)
            labels = batch['input_ids'].to(device)

            outputs = model(input_ids, attention_mask, images, labels=labels)
            loss = outputs.loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()

        avg_train_loss = total_train_loss / len(train_dataloader)
        print(f"Epoch {epoch + 1}, Average Training Loss: {avg_train_loss}")

        # Validation step
        total_val_loss = 0.0
        model.eval()
        with torch.no_grad():
            for batch in val_dataloader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                images = batch['image'].to(device)
                labels = batch['input_ids'].to(device)

                outputs = model(input_ids, attention_mask, images, labels=labels)
                loss = outputs.loss

                total_val_loss += loss.item()

        avg_val_loss = total_val_loss / len(val_dataloader)
        print(f"Epoch {epoch + 1}, Average Validation Loss: {avg_val_loss}")

## Evaluation Function

The `evaluate_model` function evaluates the trained model on a test dataset using various metrics.

- **Process**:
  - Generates reports for images in the test set.
  - Computes evaluation metrics (BLEU, METEOR, ROUGE, precision, recall, and F1 score).

## Metrics Calculation

Various functions and libraries (e.g., `Rouge`, `sentence_bleu`, `meteor_score`) are used to calculate evaluation metrics for the generated reports against reference reports.

- **calculate_scores**:
  - Calculates BLEU-1, BLEU-4, METEOR, and ROUGE scores.
  - Computes precision, recall, and F1 score for the generated reports.



In [1]:
from rouge import Rouge
from nltk.tokenize import word_tokenize
import nltk
from nltk.translate.meteor_score import meteor_score
import numpy as np
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import torch


In [5]:

nltk.download('wordnet')
nltk.download('omw-1.4')

findings = ['Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity',
            'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis',
            'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture',
            'Support Devices', 'No Finding']

def calculate_ce(reference, prediction):
    reference_set = set(reference.split())
    prediction_set = set(prediction.split())
    true_positives = len(reference_set.intersection(prediction_set))
    precision = true_positives / len(prediction_set) if prediction_set else 0
    recall = true_positives / len(reference_set) if reference_set else 0
    f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0
    return precision, recall, f1_score

def calculate_scores(references, predictions):
    bleu_scores_1 = []
    bleu_scores_4 = []
    meteor_scores_list = []
    rouge = Rouge()
    rouge_scores = {'rouge-1': [], 'rouge-2': [], 'rouge-l': []}
    precisions = []
    recalls = []
    f1_scores = []

    for ref, pred in zip(references, predictions):
        if not pred:  # Skip empty predictions
            continue

        ref_tokens = ref.split()
        pred_tokens = pred.split()

        # BLEU-1
        bleu_scores_1.append(sentence_bleu([ref_tokens], pred_tokens, weights=(1, 0, 0, 0), smoothing_function=SmoothingFunction().method1))

        # BLEU-4
        bleu_scores_4.append(sentence_bleu([ref_tokens], pred_tokens, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=SmoothingFunction().method1))

        # METEOR
        meteor_scores_list.append(meteor_score([ref_tokens], pred_tokens))  # Tokenize the inputs

        # ROUGE
        rouge_score = rouge.get_scores(pred, ref)
        for key in rouge_scores:
            rouge_scores[key].append(rouge_score[0][key]['f'])

        # Precision, Recall, F1
        precision, recall, f1_score = calculate_ce(ref, pred)
        precisions.append(precision)
        recalls.append(recall)
        f1_scores.append(f1_score)

    avg_bleu_score_1 = np.mean(bleu_scores_1) if bleu_scores_1 else 0
    avg_bleu_score_4 = np.mean(bleu_scores_4) if bleu_scores_4 else 0
    avg_meteor_score = np.mean(meteor_scores_list) if meteor_scores_list else 0
    avg_rouge_scores = {key: np.mean(value) if value else 0 for key, value in rouge_scores.items()}
    avg_precision = np.mean(precisions) if precisions else 0
    avg_recall = np.mean(recalls) if recalls else 0
    avg_f1_score = np.mean(f1_scores) if f1_scores else 0

    return avg_bleu_score_1, avg_bleu_score_4, avg_meteor_score, avg_rouge_scores, avg_precision, avg_recall, avg_f1_score

def evaluate_model(model, dataloader):
    model.eval()
    references = []
    predictions = []

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            images = batch['image'].to(device)

            generated_texts = model.generate_caption(input_ids, images, max_length=50)
            for i, generated_text in enumerate(generated_texts):
                reference_text = model.tokenizer.decode(input_ids[i], skip_special_tokens=True)
                if generated_text:  # Check if generated text is not empty
                    references.append(reference_text)
                    predictions.append(generated_text)

    bleu_scores_1, bleu_scores_4, meteor_scores, rouge_scores, avg_precision, avg_recall, avg_f1_score = calculate_scores(references, predictions)
    return bleu_scores_1, bleu_scores_4, meteor_scores, rouge_scores, avg_precision, avg_recall, avg_f1_score


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\vijay\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\vijay\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [ ]:
import os
import re  # Using the regular expression library to parse the new format
from torchvision import transforms
from sklearn.model_selection import train_test_split

# This is the main script that orchestrates the entire process.
# I've updated this block to parse your specific captions.txt format.
if __name__ == '__main__':
    # --- MODIFIED DATA LOADING (v2) ---
    
    # Define paths to your data
    image_dir = "C:/.Final_Year_Project/project/images"
    captions_path = 'C:/.Final_Year_Project/project/captions.txt'
    image_extension = '.jpg' # Assuming all your images have a .jpg extension

    all_image_paths = []
    all_reports = []

    current_img_id = None
    current_caption_parts = []
    
    print("Loading custom dataset from captions.txt...")
    
    with open(captions_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            # Check if the line starts with the ROCO_XXXXX image ID pattern followed by whitespace
            match = re.match(r'^(ROCO_\d{5})\s+(.*)', line)
            
            if match:
                # This line starts a new caption. First, save the previous one.
                if current_img_id:
                    full_path = os.path.join(image_dir, current_img_id + image_extension)
                    # Check if the image file actually exists before saving the caption
                    if os.path.exists(full_path):
                        full_caption = ' '.join(current_caption_parts)
                        all_image_paths.append(full_path)
                        all_reports.append(full_caption)

                # Now, start the new record from the matched line
                current_img_id = match.group(1)
                initial_caption_part = match.group(2).strip()
                current_caption_parts = [initial_caption_part] if initial_caption_part else []
            
            elif current_img_id:
                # If the line does not start with an ID, it's a continuation of the current caption
                current_caption_parts.append(line)

    # Make sure to save the very last record after the loop finishes
    if current_img_id:
        full_path = os.path.join(image_dir, current_img_id + image_extension)
        if os.path.exists(full_path):
            full_caption = ' '.join(current_caption_parts)
            all_image_paths.append(full_path)
            all_reports.append(full_caption)

    print(f"Loaded {len(all_image_paths)} matching image-caption pairs.")

    # Add a warning if no data was loaded to help with debugging
    if len(all_image_paths) == 0:
        print("\nWARNING: No matching image-caption pairs were found.")
        print("Please check the following:")
        print(f"1. Your 'captions.txt' file is in the correct directory.")
        print(f"2. Your image folder is named '{image_dir}'.")
        print(f"3. Your image files have the '{image_extension}' extension (e.g., ROCO_00690.jpg).")
        print(f"4. The image IDs in 'captions.txt' match the filenames in the '{image_dir}' folder.")
    
    # --- Data Splitting and Model/Dataset Initialization (remains the same) ---
    if all_image_paths:
        # Split the data into training (80%), validation (10%), and test (10%) sets
        train_image_paths, temp_image_paths, train_reports, temp_reports = train_test_split(
            all_image_paths, all_reports, test_size=0.2, random_state=42
        )
        val_image_paths, test_image_paths, val_reports, test_reports = train_test_split(
            temp_image_paths, temp_reports, test_size=0.5, random_state=42
        )
        
        print(f"Training samples: {len(train_image_paths)}")
        print(f"Validation samples: {len(val_image_paths)}")
        print(f"Test samples: {len(test_image_paths)}")

        # The rest of the script remains the same
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model = R2GenModel(model_name='t5-base', device=device)
        model.to(device)

        image_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

        train_dataset = ReportImageDataset(train_reports, train_image_paths, model.tokenizer, image_transform=image_transform)
        val_dataset = ReportImageDataset(val_reports, val_image_paths, model.tokenizer, image_transform=image_transform)
        test_dataset = ReportImageDataset(test_reports, test_image_paths, model.tokenizer, image_transform=image_transform)
        
        print("\nDatasets created successfully. Ready for training.")
    else:
        # Prevent errors if no data was loaded
        train_dataset, val_dataset, test_dataset = None, None, None
        print("\nSkipping dataset creation as no data was loaded.")

Loading custom dataset from captions.txt...
Loaded 4654 matching image-caption pairs.
Training samples: 3723
Validation samples: 465
Test samples: 466


c:\Users\vijay\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\vijay\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\vijay\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_dow


Datasets created successfully. Ready for training.


In [ ]:
train(model, train_dataset, val_dataset, batch_size=8, num_epochs=20, learning_rate=5e-5)


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch 1, Average Training Loss: 0.7287372055547432
Epoch 1, Average Validation Loss: 0.4339292269136946
Epoch 2, Average Training Loss: 0.4096870997944615
Epoch 2, Average Validation Loss: 0.4043208360671997
Epoch 3, Average Training Loss: 0.3734514497764121
Epoch 3, Average Validation Loss: 0.42422211372246177


In [9]:
torch.save(model.state_dict(), 'r2gen_model_Bert.pth')


In [10]:
test_dataloader = torch.utils.data.DataLoader(test_dataset, batch_size=8, shuffle=False)
bleu_scores_1, bleu_scores_4, meteor_scores, rouge_scores, avg_precision, avg_recall, avg_f1_score = evaluate_model(model, test_dataloader)

print("BLEU 1 Scores:", bleu_scores_1)
print("BLEU 4 Scores:", bleu_scores_4)
print("METEOR Scores:", meteor_scores)
print("ROUGE Scores:", rouge_scores)
print("Precision:", avg_precision)
print("Recall:", avg_recall)
print("F1 Score:", avg_f1_score)


BLEU 1 Scores: 0.320667658422749
BLEU 4 Scores: 0.20566399494397863
METEOR Scores: 0.4029370131605827
ROUGE Scores: {'rouge-1': 0.5356403273116664, 'rouge-2': 0.38112131084236706, 'rouge-l': 0.530092046507344}
Precision: 0.7326488569909622
Recall: 0.4187700588528864
F1 Score: 0.5147292194640016


In [11]:
import random
import torch

def print_random_captions(model, train_dataset, test_dataset, num_samples=5):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def get_random_samples(dataset, num_samples):
        indices = torch.tensor(random.sample(range(len(dataset)), num_samples)).to(dtype=torch.long, device=device)
        samples = [dataset[i] for i in indices]
        return samples

    train_samples = get_random_samples(train_dataset, num_samples)
    test_samples = get_random_samples(test_dataset, num_samples)

    model.eval()
    with torch.no_grad():
        print("Train Samples:\n")
        for sample in train_samples:
            input_ids = sample['input_ids'].unsqueeze(0).to(device)
            image = sample['image'].unsqueeze(0).to(device)
            # Check input tensor type and shape
            print(f"Input Tensor Type: {input_ids.dtype}, Shape: {input_ids.shape}")
            ground_truth = model.tokenizer.decode(input_ids[0], skip_special_tokens=True)
            generated_caption = model.generate_caption(input_ids, image)  # Corrected here
            print(f"Ground Truth: {ground_truth}")
            print(f"Generated Caption: {generated_caption}\n")

        print("Test Samples:\n")
        for sample in test_samples:
            input_ids = sample['input_ids'].unsqueeze(0).to(device)
            image = sample['image'].unsqueeze(0).to(device)
            # Check input tensor type and shape
            print(f"Input Tensor Type: {input_ids.dtype}, Shape: {input_ids.shape}")
            ground_truth = model.tokenizer.decode(input_ids[0], skip_special_tokens=True)
            generated_caption = model.generate_caption(input_ids, image)  # Corrected here
            print(f"Ground Truth: {ground_truth}")
            print(f"Generated Caption: {generated_caption}\n")
print_random_captions(model, train_dataset, test_dataset)


Train Samples:

Input Tensor Type: torch.int64, Shape: torch.Size([1, 512])
Ground Truth: generate report : lateral oblique x - ray of undisplaced body of mandible
Generated Caption: ['generate report : x - ray showing the right.']

Input Tensor Type: torch.int64, Shape: torch.Size([1, 512])
Ground Truth: generate report : abdominal x - rays showing a dilated transverse colon with a “ u - shaped ” loop in the left upper abdomen.
Generated Caption: ['generate report : x - ray of the left -.']

Input Tensor Type: torch.int64, Shape: torch.Size([1, 512])
Ground Truth: generate report : left lateral view of chest x - ray. on lateral view, the heart is displaced posteriorly with retrosternal luscency representing an anteriorly herniated lobe.
Generated Caption: ['generate report : x - ray of the left -.']

Input Tensor Type: torch.int64, Shape: torch.Size([1, 512])
Ground Truth: generate report : chest x - ray after implantation of an extracardiac cardioverter defibrillator system at the ag

In [2]:
import os
import re

def print_all_data():
    """
    Parses captions.txt and prints the filename and caption for every
    matching image found in the 'images' folder.
    """
    image_dir = "C:/.Final_Year_Project/project/images"
    captions_path = 'C:/.Final_Year_Project/project/captions.txt'
    image_extension = '.jpg'

    # --- 2. Parse the captions.txt File ---
    all_image_paths = []
    all_reports = []
    current_img_id = None
    current_caption_parts = []

    print("Parsing captions.txt to find all image-caption pairs...")
    try:
        with open(captions_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue

                match = re.match(r'^(ROCO_\d{5})\s+(.*)', line)

                if match:
                    if current_img_id:
                        full_path = os.path.join(image_dir, current_img_id + image_extension)
                        if os.path.exists(full_path):
                            all_image_paths.append(full_path)
                            all_reports.append(' '.join(current_caption_parts))

                    current_img_id = match.group(1)
                    initial_caption_part = match.group(2).strip()
                    current_caption_parts = [initial_caption_part] if initial_caption_part else []
                elif current_img_id:
                    current_caption_parts.append(line)

        # Save the last record
        if current_img_id:
            full_path = os.path.join(image_dir, current_img_id + image_extension)
            if os.path.exists(full_path):
                all_image_paths.append(full_path)
                all_reports.append(' '.join(current_caption_parts))

    except FileNotFoundError:
        print(f"Error: The file '{captions_path}' was not found.")
        return

    if not all_image_paths:
        print("Could not find any matching image-caption pairs.")
        return

    # --- 3. Print All Found Data ---
    dataset = list(zip(all_image_paths, all_reports))
    print(f"\nFound {len(dataset)} pairs. Printing all of them below:\n")
    print("=" * 70)

    for i, (path, caption) in enumerate(dataset, 1):
        print(f"[{i}/{len(dataset)}]")
        print(f"Image File: {os.path.basename(path)}")
        print(f"Caption: {caption}\n")
        print("-" * 50)

# --- Run the function ---
print_all_data()

Parsing captions.txt to find all image-caption pairs...

Found 4654 pairs. Printing all of them below:

[1/4654]
Image File: ROCO_11332.jpg
Caption: 66 y.o. male had sigmoid resection for carcinoma @ 3:00 pm. He was fed immediately for 17 hours via a double lumen, nasoduodenal catheter @ 100 kcal/hr. At 8:00 am the tube was removed, and BaSO4 swallowed. Serial X-rays indicate normal motility. 12:00 noon – 4 hour motility study, 21 hours after surgery.

--------------------------------------------------
[2/4654]
Image File: ROCO_23164.jpg
Caption: Erect abdominal X-ray reveals a large hiatal hernia with a greatly distended gastric bubble and distended bowel loops.

--------------------------------------------------
[3/4654]
Image File: ROCO_10983.jpg
Caption: Chest X-ray after whole lung lavage

--------------------------------------------------
[4/4654]
Image File: ROCO_62955.jpg
Caption: Figure 1: Plain abdominal X-rays showing a calcified mass in the pelvis.

------------------------

In [3]:
import os
import re

def verify_all_images_exist():
    """
    Checks the 'images' folder to ensure that every image mentioned in
    'captions.txt' actually exists.
    """
    # --- 1. Define Paths and Parameters ---
    image_dir = "C:/.Final_Year_Project/project/images"
    captions_path = 'C:/.Final_Year_Project/project/captions.txt'
    image_extension = '.jpg'  # Change this if your images are .png, .jpeg, etc.

    # --- 2. Extract all unique image IDs from captions.txt ---
    mentioned_ids = set()
    print(f"Reading all image IDs from '{captions_path}'...")
    try:
        with open(captions_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                
                # Use regex to find lines that start with the image ID pattern
                match = re.match(r'^(ROCO_\d{5})', line)
                if match:
                    mentioned_ids.add(match.group(1))

    except FileNotFoundError:
        print(f"Error: The file '{captions_path}' was not found in the current directory.")
        return

    if not mentioned_ids:
        print("Could not find any image IDs in 'captions.txt'. The file might be empty or in the wrong format.")
        return

    # --- 3. Check for the existence of each image file ---
    print(f"\nFound {len(mentioned_ids)} unique image IDs. Checking if they exist in the '{image_dir}' folder...")
    
    found_count = 0
    missing_files = []

    for image_id in sorted(list(mentioned_ids)): # Sort for an orderly list
        expected_filename = image_id + image_extension
        full_path = os.path.join(image_dir, expected_filename)
        
        if os.path.exists(full_path):
            found_count += 1
        else:
            missing_files.append(expected_filename)

    # --- 4. Print the Final Report ---
    print("\n" + "="*50)
    print("        Dataset Verification Report")
    print("="*50)
    print(f"Total Unique Image IDs in captions.txt: {len(mentioned_ids)}")
    print(f"Total Images Found in '{image_dir}':     {found_count}")
    print(f"Total Images Missing:                    {len(missing_files)}")
    print("="*50)

    if missing_files:
        print("\nThe following image files are MISSING from the 'images' folder:")
        for filename in missing_files:
            print(f"- {filename}")
    else:
        print("\n✅ All images mentioned in captions.txt are present in the 'images' folder.")

# --- Run the verification function ---
verify_all_images_exist()

Reading all image IDs from 'C:/.Final_Year_Project/project/captions.txt'...

Found 2327 unique image IDs. Checking if they exist in the 'C:/.Final_Year_Project/project/images' folder...

        Dataset Verification Report
Total Unique Image IDs in captions.txt: 2327
Total Images Found in 'C:/.Final_Year_Project/project/images':     2327
Total Images Missing:                    0

✅ All images mentioned in captions.txt are present in the 'images' folder.


In [4]:
import re

def count_unique_roco_ids():
    """
    Reads captions.txt and counts the number of unique ROCO_IDs.
    """
    image_dir = "C:/.Final_Year_Project/project/images"
    captions_path = 'C:/.Final_Year_Project/project/captions.txt'
    unique_ids = set()

    try:
        with open(captions_path, 'r', encoding='utf-8') as f:
            for line in f:
                # Find the first occurrence of the ROCO ID pattern in each line
                match = re.search(r'(ROCO_\d{5})', line)
                if match:
                    unique_ids.add(match.group(1))

        print(f"Total number of unique ROCO_IDs found: {len(unique_ids)}")

    except FileNotFoundError:
        print(f"Error: The file '{captions_path}' was not found.")

# --- Run the counting function ---
count_unique_roco_ids()

Total number of unique ROCO_IDs found: 2327
